In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
orders_df = (
    spark.read.table("retail_lakehouse.bronze.orders")
)
display(orders_df.limit(1))
orders_df.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders_window = Window.partitionBy("order_id")

orders_validation_df = (
    orders_df

    # Standardize source columns
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("sales_channel", F.upper(F.trim(F.col("sales_channel"))))
    .withColumn("shipping_state", F.upper(F.trim(F.col("shipping_state"))))

    # Order ID required
    .withColumn(
        "dq_order_id_not_null",
        F.col("order_id").isNotNull()
    )

    # Duplicate check
    .withColumn(
        "dq_order_id_count",
        F.count("*").over(orders_window)
    )

    .withColumn(
        "dq_order_id_unique",
        F.col("dq_order_id_count") == 1
    )

    # Customer required
    .withColumn(
        "dq_customer_id_not_null",
        F.col("customer_id").isNotNull()
    )

    # Timestamp required
    .withColumn(
        "dq_order_timestamp",
        F.col("order_timestamp").isNotNull()
    )

    # Status validation
    .withColumn(
        "dq_order_status",
        F.col("order_status").isin(
            "PLACED",
            "PAID",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        )
    )

    # Channel validation
    .withColumn(
        "dq_sales_channel",
        F.col("sales_channel").isin(
            "WEB",
            "MOBILE",
            "STORE"
        )
    )

    # Shipping state required
    .withColumn(
        "dq_shipping_state",
        F.col("shipping_state").isNotNull()
    )
)

In [0]:
orders_clean_df = (
    orders_clean_df
    .dropDuplicates()
)

In [0]:
orders_clean_df = (
    orders_validation_df
    .dropDuplicates(["order_id"])
)

In [0]:
display(
    orders_validation_df.select(
        "order_id",
        "customer_id",
        "order_timestamp",
        "order_status",
        "sales_channel",
        "shipping_state",
        "dq_order_id_not_null",
        "dq_order_id_unique",
        "dq_customer_id_not_null",
        "dq_order_timestamp",
        "dq_order_status",
        "dq_sales_channel",
        "dq_shipping_state"
    )
)

In [0]:
customer_lookup_df = (
    spark.table("retail_lakehouse.sliver.customers")
    .select(
        F.col("customer_id").alias("ref_customer_id")
    )
    .dropDuplicates()
)

In [0]:
orders_customer_check_new_df = (
    orders_validation_df
    .join(
        customer_lookup_df,
        orders_validation_df["customer_id"] ==
        customer_lookup_df["ref_customer_id"],
        "left"
    )
    .withColumn(
        "dq_customer_exists",
        F.col("ref_customer_id").isNotNull()
    )
)

In [0]:
display(
    orders_customer_check_new_df.select(
        "order_id",
        "customer_id",
        "ref_customer_id",
        "dq_customer_exists"
    )
)

In [0]:
# Overall pass/Fail step

orders_final_df = (
    orders_customer_check_new_df
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_order_id_not_null") &
            F.col("dq_order_id_unique") &
            F.col("dq_customer_id_not_null") &
            F.col("dq_customer_exists") &
            F.col("dq_order_timestamp") &
            F.col("dq_order_status") &
            F.col("dq_sales_channel") &
            F.col("dq_shipping_state"),
            "Pass"
        ).otherwise("Fail")
    )
)

In [0]:
display(orders_final_df)

In [0]:
valid_orders_df = (
    orders_final_df
    .filter(F.col("dq_status") == "Pass")
)

invalid_orders_df = (
    orders_final_df
    .filter(F.col("dq_status") == "Fail")
)

In [0]:
silver_orders_df = (
    valid_orders_df
    .drop(
        "ref_customer_id",
        "dq_order_id_count",
        "dq_order_id_not_null",
        "dq_order_id_unique",
        "dq_customer_id_not_null",
        "dq_customer_exists",
        "dq_order_timestamp",
        "dq_order_status",
        "dq_sales_channel",
        "dq_shipping_state",
        "dq_status"
    )
)

In [0]:
# write delta table
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

(
    silver_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("orders")
)

(
    invalid_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("orders_quarantine")
)